## Read raw data into pandas

In [4]:
import pandas as pd

raw = pd.read_csv("../data/raw/raw_profiles_small.csv")
raw.head()

,profile_id,country,experience
0,1,Romania,Senior LMS Technical Consultant Oracle Full-ti...
1,2,romania,Service Operations Expert\nService Operations ...
2,3,sweden,"Software Engineer"" Software Engineer Flightloo..."
3,4,sweden,"Master Thesis Researcher"" Master Thesis Resear..."
4,5,sweden,"Founding Product Designer"" UI BakeryUI Bakery ..."


## Define the experience data class
The experience data class will correspond to one row in the resulting csv.

In [5]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class Experience:
    """Represents a single work experience entry from a LinkedIn profile.
    
    Designed to map directly to a CSV row.
    """
    profile_id: int
    job_title: str
    country: Optional[str]
    start_date: Optional[str]  # e.g. "2020-01" or "Jan 2020"
    end_date: Optional[str]    # None means present/current role
    company: Optional[str] = None  # Optional field for company name
    position_in_career: Optional[int] = None  # Optional field for position in career timeline

## Format the Experiences
Format the existing experiences in the format that matches the experience data class.

In [6]:
import os
import json
import time
import re
from dotenv import load_dotenv
from google import genai
from google.genai.errors import ClientError

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)
MODEL = "gemini-2.5-flash"  # alternatives: "gemini-2.5-flash", "gemini-3-flash-preview", "gemini-2.5-flash-lite"

PARSE_PROMPT = """You are a structured data extractor. Given raw LinkedIn experience text for one person,
extract each distinct job/position into a JSON array. The text is messy — it may contain duplicated
fragments, merged lines, location strings, employment types (Full-time, Part-time, etc.), and
date strings that appear twice.

For each job, extract:
- "job_title": the cleaned job title (e.g. "Software Engineer", "IT Manager")
- "company": the company/organisation name, or null if unclear
- "start_date": in "YYYY-MM" format if month is available, or "YYYY" if only year, or null
- "end_date": in "YYYY-MM" format, "YYYY", or null if marked "Present" or still current
- "is_current": true if this is a current/present position, false otherwise

Rules:
- Ignore location strings, employment type labels (Full-time, Part-time, Hybrid, Remote, On-site),
  logo mentions, and duration strings (e.g. "6 yrs 3 mos").
- If the same position appears to be duplicated (same title, same company, same dates), output it only once.
- If a company umbrella header appears (e.g. "ING Hubs Romania Full-time · 6 yrs 9 mos"),
  treat it as context for the sub-positions below it, not as a separate job.
- Return ONLY the JSON array, no markdown fences, no explanation.

Raw experience text:
{experience_text}

JSON array:"""

REQUESTS_PER_MINUTE = 13  # stay safely under free-tier 15/min
MAX_RETRIES = 3


def parse_experience_with_llm(profile_id: int, country: str, experience_text: str) -> list[Experience]:
    """Send raw experience text to Gemini and parse the JSON response into Experience objects.
    
    position_in_career is assigned so that 1 = oldest job, N = most recent job.
    LinkedIn lists jobs most-recent-first, so the first item in the array gets the highest index.
    """
    if pd.isna(experience_text) or not experience_text.strip():
        return []

    prompt = PARSE_PROMPT.format(experience_text=experience_text)

    for attempt in range(MAX_RETRIES):
        try:
            response = client.models.generate_content(model=MODEL, contents=prompt)
            raw_text = response.text.strip()
            # Strip markdown fences if present
            raw_text = re.sub(r"^```(?:json)?\s*", "", raw_text)
            raw_text = re.sub(r"\s*```$", "", raw_text)

            jobs = json.loads(raw_text)
            n = len(jobs)
            experiences = []
            for idx, job in enumerate(jobs):
                experiences.append(Experience(
                    profile_id=profile_id,
                    job_title=job.get("job_title", "Unknown"),
                    country=country,
                    start_date=job.get("start_date"),
                    end_date=job.get("end_date") if not job.get("is_current") else None,
                    company=job.get("company"),
                    position_in_career=n - idx,  # most recent (first in list) → highest index
                ))
            return experiences

        except (json.JSONDecodeError, KeyError) as e:
            print(f"  JSON parse error for profile {profile_id} (attempt {attempt+1}): {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(5)
        except ClientError as e:
            if "429" in str(e):
                match = re.search(r"retry in (\d+)", str(e))
                wait = int(match.group(1)) + 5 if match else 60
                print(f"  Rate limited. Waiting {wait}s (attempt {attempt+1})...")
                time.sleep(wait)
            else:
                print(f"  API error for profile {profile_id}: {e}")
                break
        except Exception as e:
            print(f"  Unexpected error for profile {profile_id}: {e}")
            break

    # Fallback: return a single unknown entry so the profile isn't lost
    return [Experience(
        profile_id=profile_id,
        job_title="PARSE_ERROR",
        country=country,
        start_date=None,
        end_date=None,
        company=None,
        position_in_career=None,
    )]


# --- Run the LLM parser across all profiles ---
all_experiences: list[Experience] = []
delay = 60 / REQUESTS_PER_MINUTE

for i, row in raw.iterrows():
    pid = row["profile_id"]
    print(f"Parsing profile {pid} ({i+1}/{len(raw)})...")
    experiences = parse_experience_with_llm(pid, row["country"], row["experience"])
    all_experiences.extend(experiences)
    print(f"  -> {len(experiences)} jobs extracted")
    time.sleep(delay)

print(f"\nDone. Total experience entries: {len(all_experiences)}")
all_experiences[:5]

Parsing profile 1 (1/5)...
  -> 6 jobs extracted
Parsing profile 2 (2/5)...
  -> 3 jobs extracted
Parsing profile 3 (3/5)...
  -> 3 jobs extracted
Parsing profile 4 (4/5)...
  -> 14 jobs extracted
Parsing profile 5 (5/5)...
  -> 6 jobs extracted

Done. Total experience entries: 32


[Experience(profile_id=1, job_title='Senior LMS Technical Consultant', country='Romania', start_date='2019-01', end_date=None, company='Oracle', position_in_career=1),
 Experience(profile_id=1, job_title='Senior Technical Support Engineer', country='Romania', start_date='2017-05', end_date=None, company='Oracle Romania', position_in_career=2),
 Experience(profile_id=1, job_title='Testing Analyst', country='Romania', start_date='2015-04', end_date=None, company='TEAMNET INTERNATIONAL SA', position_in_career=3),
 Experience(profile_id=1, job_title='Technical reviewer', country='Romania', start_date='2014-03', end_date='2015-03', company='Teamnet International', position_in_career=4),
 Experience(profile_id=1, job_title='Senior Accounting Technician', country='Romania', start_date='2005-03', end_date='2014-02', company='Ciel Romania', position_in_career=5)]

## Export to csv

In [7]:
from dataclasses import asdict

output_path = "../data/processed/parsed_experiences_small.csv"

df_experiences = pd.DataFrame([asdict(e) for e in all_experiences])
df_experiences.to_csv(output_path, index=False)

print(f"Saved {len(df_experiences)} rows to {output_path}")
df_experiences.head()

Saved 32 rows to ../data/processed/parsed_experiences_small.csv


,profile_id,job_title,country,start_date,end_date,company,position_in_career
0,1,Senior LMS Technical Consultant,Romania,2019-01,NaN,Oracle,1
1,1,Senior Technical Support Engineer,Romania,2017-05,NaN,Oracle Romania,2
2,1,Testing Analyst,Romania,2015-04,NaN,TEAMNET INTERNATIONAL SA,3
3,1,Technical reviewer,Romania,2014-03,2015-03,Teamnet International,4
4,1,Senior Accounting Technician,Romania,2005-03,2014-02,Ciel Romania,5


## Categorize Job Titles

In [ ]:
df_experiences = pd.read_csv("../data/processed/parsed_experiences_small.csv")
print(f"Loaded {len(df_experiences)} rows from parsed_experiences_small.csv")

CLASSIFY_PROMPT = """Classify the following job title into one of these categories:
- Traditional Software Development
- Low-Code/No-Code Development
- Leadership/Management
- Other

Job title: {job_title}

Respond with only the category name, nothing else."""

VALID_CATEGORIES = {
    "Traditional Software Development",
    "Low-Code/No-Code Development",
    "Leadership/Management",
    "Other",
}


def classify_job(job_title: str) -> str:
    """Classify a single job title into one of the four categories using Gemini."""
    prompt = CLASSIFY_PROMPT.format(job_title=job_title)

    for attempt in range(MAX_RETRIES):
        try:
            response = client.models.generate_content(model=MODEL, contents=prompt)
            label = response.text.strip()
            return label if label in VALID_CATEGORIES else "Other"

        except ClientError as e:
            if "429" in str(e):
                match = re.search(r"retry in (\d+)", str(e))
                wait = int(match.group(1)) + 5 if match else 60
                print(f"  Rate limited. Waiting {wait}s (attempt {attempt+1})...")
                time.sleep(wait)
            else:
                print(f"  API error for '{job_title}': {e}")
                break
        except Exception as e:
            print(f"  Unexpected error for '{job_title}': {e}")
            break

    return "ERROR"


# --- Classify all parsed experiences ---
labels = []
total = len(df_experiences)

for i, row in df_experiences.iterrows():
    title = row["job_title"]

    if title == "PARSE_ERROR":
        labels.append("ERROR")
        continue

    print(f"Classifying [{i+1}/{total}] '{title}'...")
    label = classify_job(title)
    labels.append(label)
    time.sleep(delay)

df_classified = df_experiences.copy()
df_classified["lcnc_label"] = labels

output_classified = "../data/processed/classified_jobs_gemini_small.csv"
df_classified.to_csv(output_classified, index=False)

print(f"\nDone. Saved {len(df_classified)} classified entries to {output_classified}")
df_classified.head(10)


Loaded 32 rows from parsed_experiences_small.csv
Classifying [1/32] 'Senior LMS Technical Consultant'...
Classifying [2/32] 'Senior Technical Support Engineer'...
Classifying [3/32] 'Testing Analyst'...
Classifying [4/32] 'Technical reviewer'...
Classifying [5/32] 'Senior Accounting Technician'...
Classifying [6/32] 'Inginer proiectant'...
Classifying [7/32] 'Service Operations Expert'...
Classifying [8/32] 'Core Network Engineer'...
Classifying [9/32] 'Network Supervision Engineer'...
Classifying [10/32] 'Software Engineer'...
Classifying [11/32] 'Software Developer'...
Classifying [12/32] 'Remote Software Engineer Internship'...
Classifying [13/32] 'Master Thesis Researcher'...
Classifying [14/32] 'Co-founder and Chief Innovation Officer'...
Classifying [15/32] 'IT Analyst and DevOps Engineer'...
Classifying [16/32] 'IT Manager'...
Classifying [17/32] 'Student Ambassador'...
Classifying [18/32] 'Team Leader'...
Classifying [19/32] 'Marketing Communications Manager'...
Classifying [20

,profile_id,job_title,country,start_date,end_date,company,position_in_career,lcnc_label
0,1,Senior LMS Technical Consultant,Romania,2019-01,NaN,Oracle,1,Other
1,1,Senior Technical Support Engineer,Romania,2017-05,NaN,Oracle Romania,2,Other
2,1,Testing Analyst,Romania,2015-04,NaN,TEAMNET INTERNATIONAL SA,3,Other
3,1,Technical reviewer,Romania,2014-03,2015-03,Teamnet International,4,Other
4,1,Senior Accounting Technician,Romania,2005-03,2014-02,Ciel Romania,5,Other
5,1,Inginer proiectant,Romania,2003-09,2005-03,I.N.M.A.,6,Other
6,2,Service Operations Expert,romania,2023-07,NaN,Orange,1,Other
7,2,Core Network Engineer,romania,2017-04,2023-07,Orange Services,2,Other
8,2,Network Supervision Engineer,romania,2013,2017-03,Orange Romania,3,Other
9,3,Software Engineer,sweden,2024-12,NaN,Flightloop,1,Traditional Software Development
